# Accuracy versus bond dimension

This tutorial applies a random MPO to a random MPS with `apply` and measures how
the error of the compressed product falls as the output bond dimension `chi_out`
grows. As a yardstick it uses `quimb`'s contract-then-truncate, which computes
the exact product and truncates it with SVDs: close to the best a train of that
bond dimension can do.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import quimb.tensor as qtn

from src_method import apply

## The problem

A 16-site MPO with bond dimension 4 applied to an MPS with bond dimension 16: the
exact product has bond dimension 64, which is still small enough to form and
use as the reference.

In [ ]:
n_sites = 16
H = qtn.MPO_rand(n_sites, bond_dim=4, phys_dim=2, seed=0)
psi = qtn.MPS_rand_state(n_sites, bond_dim=16, phys_dim=2, seed=1)

exact = H.apply(psi, compress=False)
norm = exact.norm()
print(f"exact bond dimension: {exact.max_bond()}")

## Sweeping `chi_out`

`apply` takes and returns plain lists of site arrays, so `.arrays` unwraps the
`quimb` networks and `qtn.MatrixProductState` wraps the result back up. A fixed
`seed` keeps the random sketches, and the figure, reproducible.

In [ ]:
chis = [4, 8, 12, 16, 24, 32, 48, 64]
src_errors, svd_errors = [], []
for chi in chis:
    phi = qtn.MatrixProductState(apply(H.arrays, psi.arrays, chi_out=chi, seed=2))
    src_errors.append(phi.distance(exact) / norm)

    truncated = exact.copy()
    truncated.compress(max_bond=chi)
    svd_errors.append(truncated.distance(exact) / norm)

for chi, e_src, e_svd in zip(chis, src_errors, svd_errors):
    print(f"chi_out={chi:3d}  SRC {e_src:.2e}  SVD {e_svd:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(chis, np.maximum(src_errors, 1e-16), "o-", label="SRC (apply)")
ax.semilogy(chis, np.maximum(svd_errors, 1e-16), "s--", label="exact + SVD truncation")
ax.set_xlabel("chi_out")
ax.set_ylabel("relative error")
ax.legend()
fig.tight_layout()

Both curves fall together and reach rounding error once `chi_out` matches the
exact bond dimension of 64. SRC sits somewhat above the SVD truncation: it never
forms the exact product, and its randomized sketch is not optimal. In exchange,
its cost grows with `chi_out**2` times the input bond dimensions instead of with
the cube of the exact bond dimension, which is what makes it pay off once the
exact product no longer fits.